# Cleaning of geoapify json response

In [2]:
import json
from pathlib import Path
import pandas as pd

In [3]:
RAW_FILE = "../raw/venues_raw.json"
CLEAN_JSON = "../cleaned/venues_cleaned.json"
CLEAN_CSV = "../cleaned/venues_cleaned.csv"

with open(RAW_FILE, "r", encoding="utf-8") as file:
    raw_data = json.load(file)

print(f"Raw records loaded: {len(raw_data)}")

df = pd.json_normalize(raw_data)


Raw records loaded: 400


In [4]:
print(df.shape)
print(df.columns)
print(df.info())

(400, 392)
Index(['type', 'properties.name', 'properties.country',
       'properties.country_code', 'properties.state', 'properties.city',
       'properties.postcode', 'properties.suburb', 'properties.quarter',
       'properties.street',
       ...
       'properties.datasource.raw.min_height',
       'properties.datasource.raw.payment:cards',
       'properties.datasource.raw.roof:material',
       'properties.payment_options.cards', 'properties.datasource.raw.leisure',
       'properties.datasource.raw.max_age', 'properties.restrictions.max_age',
       'properties.datasource.raw.barrier',
       'properties.datasource.raw.fence_type',
       'properties.datasource.raw.surface'],
      dtype='str', length=392)
<class 'pandas.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Columns: 392 entries, type to properties.datasource.raw.surface
dtypes: float64(27), int64(2), object(41), str(322)
memory usage: 1.2+ MB
None


### The below code is to save the starting count

In [5]:
len(df)

400

In [17]:
print(raw_data)

[{'type': 'Feature', 'properties': {'name': 'Caffè Nero', 'country': 'United Kingdom', 'country_code': 'gb', 'state': 'England', 'city': 'City of Westminster', 'postcode': 'WC2N 5DS', 'suburb': 'Covent Garden', 'quarter': 'Westminster', 'street': 'Trafalgar Square', 'housenumber': '60-61', 'iso3166_2': 'GB-ENG', 'iso3166_2_sublevel': 'GB-WSM', 'lon': -0.1283651, 'lat': 51.5072772, 'state_code': 'ENG', 'formatted': 'Caffè Nero, 60-61 Trafalgar Square, London, WC2N 5DS, United Kingdom', 'address_line1': 'Caffè Nero', 'address_line2': '60-61 Trafalgar Square, London, WC2N 5DS, United Kingdom', 'categories': ['catering', 'catering.cafe', 'catering.cafe.coffee', 'catering.cafe.coffee_shop', 'vegan', 'vegetarian'], 'details': ['details', 'details.catering', 'details.facilities', 'details.payment'], 'datasource': {'sourcename': 'openstreetmap', 'attribution': '© OpenStreetMap contributors', 'license': 'Open Database License', 'url': 'https://www.openstreetmap.org/copyright', 'raw': {'lat': 51

### Save the fields that we actually need

In [ ]:
columns2 = [
    "properties.place_id",
    "properties.name",
    "properties.categories",
    "properties.formatted",
    "properties.postcode",
    "properties.city",
    "properties.lat",
    "properties.lon",
    "properties.website",
    "properties.opening_hours"
]

df = df.reindex(columns=columns2) # Makes the dataframes columns match the order and names in the columns2 list

### Rename the columns to match the database fields:

In [ ]:
df = df.rename(columns={
    "properties.place_id": "geoapify_place_id",
    "properties.name": "name",
    "properties.categories": "categories",
    "properties.formatted": "address",
    "properties.postcode": "postcode",
    "properties.city": "city",
    "properties.lat": "latitude",
    "properties.lon": "longitude",
    "properties.website": "website",
    "properties.opening_hours": "opening_hours"
})

### Clean the category field to match the database

- Database schema only has one category field wheras API returns multiple.

In [13]:
def get_category(categories):
    if not isinstance(categories, list):
        return None

    wanted_prefixes = [
        "catering.cafe",
        "catering.restaurant",
        "entertainment.museum",
        "leisure.playground"
    ]

    for prefix in wanted_prefixes:
        for category in categories:
            if category.startswith(prefix):
                return category

    return None

df["category"] = df["categories"].apply(get_category)
df.sample(50)

,geoapify_place_id,name,categories,address,postcode,borough,latitude,longitude,website,opening_hours,category
247,513424383f0fffbcbf596aedc73725c34940f00103f901...,Mail Rail Museum,"[entertainment, entertainment.museum, fee, whe...","Mail Rail Museum, Phoenix Place, London, WC1X ...",WC1X 0BF,London Borough of Islington,51.524573,-0.113267,https://www.postalmuseum.org/,We-Su 10:00-17:00,entertainment.museum
256,51d2b30f689f7dc3bf59729307d8fdc24940f00103f901...,Royal Academy of Music Museum,"[entertainment, entertainment.museum]","Royal Academy of Music Museum, Marylebone Road...",NW1 5HT,NaN,51.523372,-0.152271,https://www.ram.ac.uk/museum,Fr 11:00-18:00,entertainment.museum
185,518ae76c01a1f5c0bf59cabf3b9e3bc14940f00103f901...,Tiger Tiger,"[catering, catering.restaurant]","Tiger Tiger, 29 Haymarket, London, SW1Y 4SP, U...",SW1Y 4SP,St. James's,51.509632,-0.132496,https://london.tigertiger.co.uk/,NaN,catering.restaurant
199,516d24647b88a1c0bf5952b5824b6cc14940f00103f901...,Petit Bistro,"[catering, catering.restaurant]","Petit Bistro, 14 Leicester Square, London, WC2...",WC2H 7NG,St. James's,51.511117,-0.129930,NaN,NaN,catering.restaurant
265,5153848e12aaa8b6bf5969e1ef32a5c04940f00102f901...,Old Operating Theatre Museum,"[building, building.tourism, entertainment, en...","Old Operating Theatre Museum, 9a St. Thomas St...",SE1 9RY,London Borough of Southwark,51.505025,-0.088520,https://oldoperatingtheatre.com/,Mo-Su 10:30-17:00,entertainment.museum
18,519fc9fe791a30c0bf5913c0289c38c14940f00103f901...,Rosetta,"[catering, catering.cafe]","Rosetta, 38 William IV Street, London, WC2B 4D...",WC2B 4DD,NaN,51.509540,-0.126468,https://www.facebook.com/rosettafarmlondon,Mo-Sa 10:30-19:30,catering.cafe
116,5165ecc4001dc9bfbf595654a35812c14940f00102f901...,Hobson Fish & Chips,"[catering, catering.restaurant]","Hobson Fish & Chips, 13-15 Villiers Street, Lo...",WC2N 6ND,NaN,51.508372,-0.124162,NaN,NaN,catering.restaurant
164,51447cbac0b568c0bf5990f30e0861c14940f00102f901...,Street Burger,"[catering, catering.restaurant, catering.resta...","Street Burger, 24 Charing Cross Road, London, ...",WC2H 0HX,NaN,51.510779,-0.128197,https://www.gordonramsayrestaurants.com/en/uk/...,Mo-Su 11:30-22:00; Th-Sa 11:30-23:00,catering.restaurant
213,517534754923bcbcbf594f2f3e067ec14940f00102f901...,Two Temple Place,"[building, building.tourism, entertainment, en...","Two Temple Place, 2 Temple Place, London, WC2R...",WC2R 3BD,St Clement Danes,51.511644,-0.112200,https://twotempleplace.org/,NaN,entertainment.museum
27,510c5634eb3439c0bf592e53b0c447c14940f00102f901...,Caffè Nero,"[building, building.catering, catering, cateri...","Caffè Nero, 36A St Martin's Lane, London, WC2N...",WC2N 4ER,NaN,51.510005,-0.126744,https://www.caffenero.com/uk/stores/st-martins...,Mo-Fr 07:00-20:30; Sa 07:30-20:30; Su 08:00-19:00,catering.cafe


### Removal of null values
- The below code shows the venues that have null values and the cell block after that removes these values.


In [9]:
df[df['latitude'].isnull() | df['longitude'].isnull() | df['name'].isnull() | df['geoapify_place_id'].isnull()]

,geoapify_place_id,name,categories,address,postcode,borough,latitude,longitude,website,opening_hours,category
157,512d4c658a700cc1bf597f446cd3f8c04940f00102f901...,NaN,"[catering, catering.restaurant]","12 Waterloo Place, London, SW1Y 4AR, United Ki...",SW1Y 4AR,St. James's,51.507594,-0.133192,NaN,NaN,catering.restaurant
300,51175b54f4e196bebf59c9dfda4c38c14940f00102f901...,NaN,"[access, access.yes, leisure, leisure.playgrou...","Victoria Embankment, London, WC2N 6PB, United ...",WC2N 6PB,St Clement Danes,51.509489,-0.119522,NaN,NaN,leisure.playground
302,5168cf09f87f74c0bf5915645a1ee7c14940f00102f901...,NaN,"[access, access.yes, leisure, leisure.playground]","St. Giles Passage, London, WC2H 8DE, United Ki...",WC2H 8DE,London Borough of Camden,51.514884,-0.128565,NaN,NaN,leisure.playground
304,5115ff412497cabebf59dd9d0053d0c14940f00102f901...,NaN,"[leisure, leisure.playground]","Drury Lane, London, WC2B 5SQ, United Kingdom",WC2B 5SQ,St Clement Danes,51.514170,-0.120279,NaN,NaN,leisure.playground
305,515806646c79b6c1bf59e6cda02124c04940f00102f901...,NaN,"[access, access.yes, leisure, leisure.playground]","Birdcage Walk, London, SW1H 9AP, United Kingdom",SW1H 9AP,NaN,51.501100,-0.138368,NaN,NaN,leisure.playground
...,...,...,...,...,...,...,...,...,...,...,...
395,514411d9ac9fa3b9bf59c217265305bf4940f00102f901...,NaN,"[leisure, leisure.playground]","Hampton Street, London, SE1 6SL, United Kingdom",SE1 6SL,London Borough of Southwark,51.492350,-0.100153,NaN,NaN,leisure.playground
396,51b28a2a01f96cbbbf5923e1f0d153c34940f00102f901...,NaN,"[leisure, leisure.playground]","Northampton Road, London, EC1R 0HU, United Kin...",EC1R 0HU,London Borough of Islington,51.526011,-0.107122,NaN,NaN,leisure.playground
397,51d73638d5f2bcb8bf59a4fc65017bc24940f00102f901...,NaN,"[access_limited, access_limited.private, leisu...","Thomas More Highwalk, City of London, EC2Y 8BT...",EC2Y 8BT,NaN,51.519379,-0.096633,NaN,NaN,leisure.playground
398,5197497b24d37ac2bf59a1f5251852be4940f00102f901...,NaN,"[leisure, leisure.playground]","Lupus Street, London, SW1V 3HD, United Kingdom",SW1V 3HD,NaN,51.486935,-0.144378,NaN,NaN,leisure.playground


In [10]:
df = df.dropna(subset=[
    'latitude',
    'longitude',
    'name',
    'geoapify_place_id',
    'postcode'
])

- Now the same code that checks for null values returns no results.

In [11]:
df[df['latitude'].isnull() | df['longitude'].isnull() | df['name'].isnull() | df['geoapify_place_id'].isnull()]

,geoapify_place_id,name,categories,address,postcode,borough,latitude,longitude,website,opening_hours,category


- No duplicated valeus:

In [12]:
df[df.duplicated(keep=False)]

TypeError: unhashable type: 'list'

### Clean venue names
- Remove white spaces in front or trailing

In [ ]:
df["name"] = df["name"].str.strip()

### Clean postcodes

In [ ]:
df["postcode"] = (
    df["postcode"].str.strip().str.upper()
)

In [ ]:
print(df.info())
print(df.shape)
print(len(df))

<class 'pandas.DataFrame'>
Index: 318 entries, 0 to 394
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   geoapify_place_id  318 non-null    str    
 1   name               318 non-null    str    
 2   address            318 non-null    str    
 3   postcode           318 non-null    string 
 4   borough            197 non-null    str    
 5   latitude           318 non-null    float64
 6   longitude          318 non-null    float64
 7   website            179 non-null    str    
 8   opening_hours      112 non-null    str    
 9   category           318 non-null    str    
dtypes: float64(2), str(7), string(1)
memory usage: 27.3 KB
None
(318, 10)
318


### Export cleaned data to the cleaned data folder:

In [ ]:
df.to_json(
    CLEAN_JSON,
    orient="records",
    indent=2,
    force_ascii=False
)

df.to_csv(
    CLEAN_CSV,
    index=False
)